# Runes, strings and bytes

## Decyphering runes

Just as C has a `char` type that is a number that can be displayed as a character,
Go has its `rune` type (a nod to [runes](https://en.wikipedia.org/wiki/Rune)).

In C, a `char` is usually one byte, and may or may not be signed. Conventionally, bytes from `0x00` to `0x7f` correspond to characters in the [ASCII standard](https://en.wikipedia.org/wiki/ASCII).

In Go, a `rune` takes 4 bytes, is signed, and its value is a _codepoint_ in the [Unicode standard](https://home.unicode.org/).

A Unicode _codepoint_ is an integer from `0` to `0x10ffff` that represents a character from a human language or an emoji (but many codepoints are not yet assigned, and others are special markers that are not graphical symbols).

The `rune` type is an _alias_ for `int32`, conceptually the same as:

```go
type rune = int32
```

Example showing a `rune` representing the grinning cat emoji, in 4 different ways:

In [1]:
%%
cat := '\U0001F638'
fmt.Printf("%T\t%d\t%U\t%q\n", cat, cat, cat, cat )

int32	128568	U+1F638	'😸'


> "But wait, what `%%` syntax is this?"

Glad you asked! This is a Jupyter Notebook running the
[GONB](https://github.com/janpfeifer/gonb) kernel,
which compiles Go programs or code fragments on the fly.

Every code cell in this notebook is Go code.

The magic instruction `%%` places the rest of the cell inside a `func main {...}` and also handles the necessary `import` statements to make it work.

Note in the example:

* `'\U0001F638'` is a rune literal;
* The formats displayed are:
  * `%T`: data type `int32`;
  * `%d`: numeric value in decimal `128568`;
  * `%U`: codepoint in the standard Unicode format `U+1F638`;
  * `%q`: the rune in single quotes `'😸'`

This example shows several literal formats for runes:

In [2]:
package main

import "fmt"

func main() {
	runes := []rune{'A', 97, 'ç', 0x3BB, '\u6c23', '\U0001d11e'}
    fmt.Println("rune\tdecimal\tcodepoint")
    for _, r := range runes {
        fmt.Printf("%3q\t%7d\t%U\n", r, r, r)
    }
}

rune	decimal	codepoint
'A'	     65	U+0041
'a'	     97	U+0061
'ç'	    231	U+00E7
'λ'	    955	U+03BB
'氣'	  27683	U+6C23
'𝄞'	 119070	U+1D11E


## A string is not a sequence of runes

In Python, the expression `"café"[3]` returns the letter `é`, codepoint U-00E9.

In Python's `str` class, each item is a Unicode codepoint.

Now look at this example in Go:

In [3]:
%%
x := "café"[3]
fmt.Printf("%x %q", x, x)

c3 'Ã'

Accessing a string by index returns one byte at a time.

The `string` type in Go is an immutable sequence of bytes.

The `%q` verb in `fmt` displays the byte `"café"[3]` as the rune U-00C3, which is `'Ã'` (uppercase A with a tilde), because its value is `c3`. But in reality this is only half of the 2-byte sequence that represents `'é'`: `c3 a9`.

See the result of `len`:

In [4]:
%%
fmt.Println(len("café"))

5


When converting each `rune` to `string`, the result can be 1, 2, 3, or 4 bytes:

In [5]:
var runes = []rune{'A', 97, 'é', 0x3BB, '\u6c23', '\U0001d11e'}

%%
fmt.Println("rune\tdecimal\tcodepoint\tstring\tlen(s)")
for _, r := range runes {
    s := string(r)
    fmt.Printf("%3q\t%7d\t%U\t\t%q\t% x\n", r, r, r, s, len(s))
}


rune	decimal	codepoint	string	len(s)
'A'	     65	U+0041		"A"	 1
'a'	     97	U+0061		"a"	 1
'é'	    233	U+00E9		"é"	 2
'λ'	    955	U+03BB		"λ"	 2
'氣'	  27683	U+6C23		"氣"	 3
'𝄞'	 119070	U+1D11E		"𝄞"	 4


### A string may be a sequence of UTF-8 bytes

The variable size of the `rune`-to-`string` conversion is a feature of UTF-8 encoding: an algorithm for converting codepoints to bytes.

There are other algorithms, such as UTF-16 and UTF-32, but UTF-8 is the most widely used. It is the default in the Go language.

In this example we can see the bytes that represent each codepoint as UTF-8 within a string:

In [6]:
%%
fmt.Println("\trune\tlen(s)\tUTF-8 bytes")
for _, r := range runes {
    s := string(r)
	fmt.Printf("%U\t %q\t%3d\t% x\n", r, r, len(s), s)
}

	rune	len(s)	UTF-8 bytes
U+0041	 'A'	  1	41
U+0061	 'a'	  1	61
U+00E9	 'é'	  2	c3 a9
U+03BB	 'λ'	  2	ce bb
U+6C23	 '氣'	  3	e6 b0 a3
U+1D11E	 '𝄞'	  4	f0 9d 84 9e


In Go, `string` is a sequence of bytes that frequently (but not always) represents Unicode text encoded as UTF-8.

If the `string` was created from a string literal `"Hello"`, then it is certainly UTF-8 text because the Go compiler only accepts source code in UTF-8.

But if the `string` was read from a file, or created in some other way, it may not be UTF-8 text.

Here is a string with random bytes:

In [7]:
%%
b := make([]byte, 20)
rand.Read(b)
s := string(b)
fmt.Printf("% x\n", s)
fmt.Println(s)

ba 82 b9 11 ac f7 5d fe 7c 41 7d 73 7e 5a ea b0 2a be 03 f2
�����]�|A}s~Z��*��


The symbol � is the Unicode replacement character, used when an application needs to show an unknown, unrecognised, or unrepresentable character.

## Differences between `string` and `[]byte`

Given that `string` is really a sequence of bytes, how is it different from `[]byte` (a _slice_ of bytes)?

### Mutability

Strings are immutable. In contrast, `[]byte` is a _slice_, so it can grow and we can change any byte in it.

### Size

In both cases, `len()` returns the number of bytes. This operation is O(1) because the value is already stored in the internal struct that represents these objects.

To count the number of runes in a `string` (assuming the content is UTF-8), use `utf8.RuneCountInString(s)`.
This is O(n) because the function needs to traverse the entire `string` to decode and count the runes.

In [8]:
%%
fmt.Println(utf8.RuneCountInString("café"))

4


### Iteration

The `for/range` loop has special handling for `string`.

Go assumes that a `string` is a sequence of runes encoded in UTF-8.

So during iteration we get:

* the byte index where a UTF-8 character begins;
* the corresponding rune.

See `i` and `x` in this example:

In [9]:
var s = "Olá!"

%%
for i, x := range(s) {
    fmt.Printf("%d %T %U %q\n", i, x, x, x)
}

0 int32 U+004F 'O'
1 int32 U+006C 'l'
2 int32 U+00E1 'á'
4 int32 U+0021 '!'


Note how the value of `i` jumps from 2 to 4, because the rune `'á'` occupies two bytes in UTF-8:

But when iterating over `[]byte`, we get each individual byte:

In [10]:
%%
b := []byte(s)
for i, x := range(b) {
    fmt.Printf("%d %T %x\n", i, x, x)
}

0 uint8 4f
1 uint8 6c
2 uint8 c3
3 uint8 a1
4 uint8 21


## Midnight oil

> The **Midnight oil** section is optional content: curiosities, bit-twiddling, and random deep dives.

### Not every codepoint is a "character"

Each `rune` represents a codepoint, but not every codepoint represents a visible character.

Some are special markers. For example, there are no codepoints that directly represent national flags.
There are 26 "REGIONAL INDICATOR SYMBOL" codepoints associated with the letters A through Z,
which form flags when shown in pairs.

The Brazilian flag is formed by the sequence '🇧', '🇷':

In [11]:
%%
s := "🇧🇷"
fmt.Println("len =", len(s))
fmt.Println("rune count =", utf8.RuneCountInString(s))
for _, x := range(s) {
    fmt.Printf("%U %q\n", x, x)
}


len = 8
rune count = 2
U+1F1E7 '🇧'
U+1F1F7 '🇷'


### `string` and `[]byte` in memory 


In memory, the `string` type is represented by an internal _struct_ with two fields: `len` (the current number of bytes) and a pointer to the content bytes.

A `[]byte` byte slice, like all slices, is a struct in memory with three fields: `len`, `cap` (the maximum capacity), and a pointer to a byte array managed by the Go _runtime_.

### Arithmetic with runes

Since runes are numbers, we can use arithmetic to find numerically related characters.

For example, the faces of a D6 die:

In [12]:
var die1 rune = '\u2680'

%%
fmt.Printf("%d  %U  %c\n", die1, die1, die1)

die2 := die1 + 1
fmt.Printf("%d  %U  %c\n", die2, die2, die2)

die6 := die1 + 5
fmt.Printf("%d  %U  %c\n", die6, die6, die6)

9856  U+2680  ⚀
9857  U+2681  ⚁
9861  U+2685  ⚅


Or the eight [trigrams](https://en.wikipedia.org/wiki/Bagua):

In [13]:
%%
trigramHeaven := '\u2630'
for i := rune(0); i <8; i++ {
    fmt.Printf("%c ", trigramHeaven + i)
}

☰ ☱ ☲ ☳ ☴ ☵ ☶ ☷ 

Or all the flags supported on your system, by generating all combinations of REGIONAL INDICATOR SYMBOLs

In [14]:
%%
var regionalA, i, j rune
regionalA = '\U0001F1E6'  // REGIONAL INDICATOR SYMBOL LETTER A
for i = range(26) {
    fmt.Printf("%c:", 'A' + i)
    for j = range(26) {
        flag := string([]rune{regionalA+i, regionalA+j})
        fmt.Print(flag, "|")
    }
    fmt.Println()
}

A:🇦🇦|🇦🇧|🇦🇨|🇦🇩|🇦🇪|🇦🇫|🇦🇬|🇦🇭|🇦🇮|🇦🇯|🇦🇰|🇦🇱|🇦🇲|🇦🇳|🇦🇴|🇦🇵|🇦🇶|🇦🇷|🇦🇸|🇦🇹|🇦🇺|🇦🇻|🇦🇼|🇦🇽|🇦🇾|🇦🇿|
B:🇧🇦|🇧🇧|🇧🇨|🇧🇩|🇧🇪|🇧🇫|🇧🇬|🇧🇭|🇧🇮|🇧🇯|🇧🇰|🇧🇱|🇧🇲|🇧🇳|🇧🇴|🇧🇵|🇧🇶|🇧🇷|🇧🇸|🇧🇹|🇧🇺|🇧🇻|🇧🇼|🇧🇽|🇧🇾|🇧🇿|
C:🇨🇦|🇨🇧|🇨🇨|🇨🇩|🇨🇪|🇨🇫|🇨🇬|🇨🇭|🇨🇮|🇨🇯|🇨🇰|🇨🇱|🇨🇲|🇨🇳|🇨🇴|🇨🇵|🇨🇶|🇨🇷|🇨🇸|🇨🇹|🇨🇺|🇨🇻|🇨🇼|🇨🇽|🇨🇾|🇨🇿|
D:🇩🇦|🇩🇧|🇩🇨|🇩🇩|🇩🇪|🇩🇫|🇩🇬|🇩🇭|🇩🇮|🇩🇯|🇩🇰|🇩🇱|🇩🇲|🇩🇳|🇩🇴|🇩🇵|🇩🇶|🇩🇷|🇩🇸|🇩🇹|🇩🇺|🇩🇻|🇩🇼|🇩🇽|🇩🇾|🇩🇿|
E:🇪🇦|🇪🇧|🇪🇨|🇪🇩|🇪🇪|🇪🇫|🇪🇬|🇪🇭|🇪🇮|🇪🇯|🇪🇰|🇪🇱|🇪🇲|🇪🇳|🇪🇴|🇪🇵|🇪🇶|🇪🇷|🇪🇸|🇪🇹|🇪🇺|🇪🇻|🇪🇼|🇪🇽|🇪🇾|🇪🇿|
F:🇫🇦|🇫🇧|🇫🇨|🇫🇩|🇫🇪|🇫🇫|🇫🇬|🇫🇭|🇫🇮|🇫🇯|🇫🇰|🇫🇱|🇫🇲|🇫🇳|🇫🇴|🇫🇵|🇫🇶|🇫🇷|🇫🇸|🇫🇹|🇫🇺|🇫🇻|🇫🇼|🇫🇽|🇫🇾|🇫🇿|
G:🇬🇦|🇬🇧|🇬🇨|🇬🇩|🇬🇪|🇬🇫|🇬🇬|🇬🇭|🇬🇮|🇬🇯|🇬🇰|🇬🇱|🇬🇲|🇬🇳|🇬🇴|🇬🇵|🇬🇶|🇬🇷|🇬🇸|🇬🇹|🇬🇺|🇬🇻|🇬🇼|🇬🇽|🇬🇾|🇬🇿|
H:🇭🇦|🇭🇧|🇭🇨|🇭🇩|🇭🇪|🇭🇫|🇭🇬|🇭🇭|🇭🇮|🇭🇯|🇭🇰|🇭🇱|🇭🇲|🇭🇳|🇭🇴|🇭🇵|🇭🇶|🇭🇷|🇭🇸|🇭🇹|🇭🇺|🇭🇻|🇭🇼|🇭🇽|🇭🇾|🇭🇿|
I:🇮🇦|🇮🇧|🇮🇨|🇮🇩|🇮🇪|🇮🇫|🇮🇬|🇮🇭|🇮🇮|🇮🇯|🇮🇰|🇮🇱|🇮🇲|🇮🇳|🇮🇴|🇮🇵|🇮🇶|🇮🇷|🇮🇸|🇮🇹|🇮🇺|🇮🇻|🇮🇼|🇮🇽|🇮🇾|🇮🇿|
J:🇯🇦|🇯🇧|🇯🇨|🇯🇩|🇯🇪|🇯🇫|🇯🇬|🇯🇭|🇯🇮|🇯🇯|🇯🇰|🇯🇱|🇯🇲|🇯🇳|🇯🇴|🇯🇵|🇯🇶|🇯🇷|🇯🇸|🇯🇹|🇯🇺|🇯🇻|🇯🇼|🇯🇽|🇯🇾|🇯🇿|
K:🇰🇦|🇰🇧|🇰🇨|🇰🇩|🇰🇪|🇰🇫|🇰🇬|🇰🇭|🇰🇮|🇰🇯|🇰🇰|🇰🇱|🇰🇲|🇰🇳|🇰🇴|🇰🇵|🇰🇶|🇰🇷|🇰🇸|🇰🇹|🇰🇺|🇰🇻|🇰🇼|🇰🇽|🇰🇾|🇰🇿|
L:🇱🇦|🇱🇧|🇱🇨|🇱🇩|🇱🇪|🇱🇫|🇱🇬|🇱🇭|🇱🇮|🇱🇯|🇱🇰|🇱🇱|🇱🇲|🇱🇳|🇱🇴|🇱🇵|🇱🇶|🇱🇷|🇱🇸|🇱🇹|🇱🇺|🇱🇻|🇱🇼|🇱🇽|🇱🇾|🇱🇿|
M:🇲🇦|🇲🇧|🇲🇨|🇲🇩|🇲🇪|🇲🇫|🇲🇬|🇲🇭|🇲🇮

## References

* [Strings, bytes, runes and characters in Go](https://go.dev/blog/strings) by Rob Pike
* Interesting discussion about `runes` as an alias for `int32`: [Go issue #29012](https://github.com/golang/go/issues/29012)
* [Counting characters in golang string](https://stackoverflow.com/questions/36928185/counting-characters-in-golang-string) (StackOverflow)
